In [ ]:
from __future__ import absolute_import, print_function
# add external modules to PYTHONPATH via environment variable
import os, sys, time
from PIL import Image

import numpy as np
from matplotlib import pyplot as plt
from collections import namedtuple
import tvm
from tvm import te
from tvm import rpc, autotvm, relay
from tvm.contrib import graph_runtime, utils, download
from tvm.contrib.debugger import debug_runtime
from tvm.relay import transform

import vta
from vta.testing import simulator
from vta.top import graph_pack

import torch
import torchvision
from tvm.contrib.download import download_testdata

# Robust import of external ofa_base_models without being shadowed by local folder
external_repo_root = "/home/srchand/Desktop/research/OFA_Obfs"
if external_repo_root not in sys.path:
    sys.path.insert(0, external_repo_root)

# If a local shim package is already cached, evict it to allow importing the external one
_mod = sys.modules.get("ofa_base_models")
if _mod is not None:
    try:
        _mod_file = getattr(_mod, "__file__", "") or ""
        if "TVM_Intel_Fork/tvm/vta/sri_scripts/jupyter_nbs/ofa_base_models" in _mod_file:
            del sys.modules["ofa_base_models"]
    except Exception:
        # If anything goes wrong, clear the cache entry
        sys.modules.pop("ofa_base_models", None)

try:
    from ofa_base_models import OFADynamicResnetAllMod  # type: ignore
    from ofa_base_models.ofa_ops.dynamic_conv_all import DynamicConv2DAll  # type: ignore
except (ModuleNotFoundError, ImportError):
    # Final fallback: ensure external root is first in path and retry once
    if sys.path[0] != external_repo_root:
        sys.path.insert(0, external_repo_root)
    # Clear any cached partial import
    sys.modules.pop("ofa_base_models", None)
    from ofa_base_models import OFADynamicResnetAllMod  # type: ignore
else:
    # Print where the module is loaded from for debugging/IDE clarity
    import ofa_base_models as _obm  # type: ignore
    print("ofa_base_models loaded from:", getattr(_obm, "__file__", None))

import torch

from torchvision import transforms


# Make sure that TVM was compiled with RPC=1
assert tvm.runtime.enabled("rpc")

In [ ]:
Workload = namedtuple(
    "Conv2DWorkload",
    [
        "batch",
        "height",
        "width",
        "in_filter",
        "out_filter",
        "hkernel",
        "wkernel",
        "hpad",
        "wpad",
        "hstride",
        "wstride",
    ],
)

In [ ]:
import re
channels_re = re.compile('.*Tensor\[\(([\d]+), ([\d]+), [\d]+, [\d]+\).*padding.*Tensor\[\([\d]+, [\d]+, ([\d]+), ([\d]+)\).*')
cast_re = re.compile('cast.*Tensor\[\([\d]+, [\d]+, ([\d]+), ([\d]+)\).*')

In [ ]:
from torchinfo import summary

net = OFADynamicResnetAllMod()
model_path = "/mnt/hgfs/vmware_ubuntu_sf/OFA_networks/all_mod_aggressive_reg_final_checkpoint.pth"
device_temp = torch.device('cpu')
checkpoint = torch.load(model_path, map_location=device_temp)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state = checkpoint['model_state_dict']
else:
    state = checkpoint
# Allow mismatched keys due to local shim modules
net.load_state_dict(state, strict=False)

summary(net, input_size=(1, 3, 224, 224))

In [ ]:
arch = net.sample_arch()
print(arch)
net.set_active_subnet(arch)
# net.precompute_active_weights(arch)
# net.enable_auto_precompute()
summary(net, input_size=(1, 3, 224, 224))

In [ ]:
import json
from typing import Dict, Any

candidate_set_json_path="/mnt/hgfs/vmware_ubuntu_sf/OFA_networks/candidate_set_final/candidates_25_85.json"
arch_config_json_path="/mnt/hgfs/vmware_ubuntu_sf/OFA_networks/candidate_set_final/architectures_20250927_180844.json"

def load_arch_mapping(path: str) -> Dict[str, Any]:
    """Load architectures JSON and normalize to a mapping {id: architecture_dict}.

    Supported input formats:
    - A dict mapping id -> architecture dict (legacy)
    - A dict with key 'architectures' containing a list of items with fields 'id' and 'architecture'
    - A top-level list of items with 'id' and 'architecture'
    """
    with open(path, 'r') as f:
        data = json.load(f)

    # Case 1: already a mapping from id -> arch
    if isinstance(data, dict) and all(isinstance(v, dict) and 'residual_depth_list' in v for v in data.values()):
        return data

    # Case 2: top-level dict with 'architectures' list
    if isinstance(data, dict) and 'architectures' in data and isinstance(data['architectures'], list):
        mapping = {}
        for item in data['architectures']:
            # item may contain fields 'id' and 'architecture' (nested)
            if 'id' in item and 'architecture' in item:
                mapping[item['id']] = item['architecture']
            elif 'id' in item and 'arch' in item:
                mapping[item['id']] = item['arch']
            else:
                # If the item itself is an architecture dict without id, generate an id
                if 'id' in item:
                    mapping[item['id']] = item
        return mapping

    # Case 3: top-level list of architecture items
    if isinstance(data, list):
        mapping = {}
        for idx, item in enumerate(data):
            if isinstance(item, dict) and 'id' in item and 'architecture' in item:
                mapping[item['id']] = item['architecture']
            elif isinstance(item, dict) and 'id' in item:
                mapping[item['id']] = item
            else:
                mapping[f'arch_{idx}'] = item
        return mapping

    # Fallback: unknown format, raise error
    raise ValueError(f"Unsupported architecture file format: {path}")

arch_mapping = load_arch_mapping(arch_config_json_path)
# read candidate set json
with open(candidate_set_json_path, 'r') as f:
    candidate_set = json.load(f)
model_ids = candidate_set['selected_model_ids']


In [ ]:
def extract_wkls(ofa_net, arch_mapping, model_id):
    pytorch_model = ofa_net
    pytorch_model.set_active_subnet(arch_mapping[model_id])
    # pytorch_model.precompute_active_weights(arch_mapping[model_id])
    pytorch_model.eval()
    # for mod in pytorch_model.modules():
    #     if hasattr(mod, 'export_detach_cached_filters'):
    #         mod.export_detach_cached_filters = True
    workloads = []
    count = 0
    input_shape = [1, 3, 224, 224]
    input_data = torch.randn(input_shape)
    pytorch_model(input_data)
    for layer in pytorch_model.modules():
        if type(layer) == torch.nn.modules.conv.Conv2d:
            if(layer.in_channels % 16 == 0 and layer.out_channels % 16 ==0 and layer.padding[0] == layer.padding[1]):
                workloads.append(Workload(1, 0, 0, layer.in_channels, layer.out_channels,
                                  layer.kernel_size[0], layer.kernel_size[1], layer.padding[0], layer.padding[1]
                                 , layer.stride[0], layer.stride[1]))
        elif type(layer) == DynamicConv2DAll:
            # print the active channels and kernel sizes and strides and paddings
            active_in_channels = layer.exec_in_channels
            active_out_channels = layer.exec_out_channels
            padding = layer.exec_padding
            kernel_size = layer.exec_kernel_size
            stride = layer.base_conv.stride

            for ic, oc in zip(active_in_channels, active_out_channels):
                print("Found sub-conv of DynamicConv2DAll with in_channels={}, out_channels={}, kernel_size={}, stride={}, padding={}".format(ic, oc, kernel_size, stride, padding))
                if(ic % 16 == 0 and oc % 16 ==0):
                    workloads.append(Workload(1, 0, 0, ic, oc,
                                      kernel_size, kernel_size, padding, padding
                                     , stride, stride))


    scripted_model = torch.jit.trace(pytorch_model, input_data).eval()
    shape_list = [("input0", input_shape)]
    mod, params = relay.frontend.from_pytorch(scripted_model, shape_list)
    # print(mod.astext(show_meta_data=False))

    # Ensure types are inferred before quantization
    mod = relay.transform.InferType()(mod)

    # Perform quantization - let quantize() handle parameter binding internally
    with tvm.transform.PassContext(opt_level=3):
        with relay.quantize.qconfig(global_scale=8.0, skip_conv_layers=[0]):
             mod = relay.quantize.quantize(mod, params=params)
    mod_as_string = mod.astext(show_meta_data=False)
    cast_line = ""
    cast_line_idx = -1
    final_workloads = []
    for i, line in enumerate(mod_as_string.split('\n')):
        if "cast" in line and "int8" in line:
            cast_line = line
            cast_line_idx = i
        elif "conv2d" in line and "int8" in line:
            match = re.search(channels_re, line)
            if match:
                if int(match.group(1)) % 16 == 0 and int(match.group(2)) % 16 == 0:
                    match_cast = re.search(cast_re, cast_line)
                    if match_cast:
                        wkl = workloads[count]
                        final_workloads.append(
                        'Workload({}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {})'.format(
                        1, match_cast.group(1), match_cast.group(2), wkl.in_filter, wkl.out_filter,
                        wkl.hkernel,wkl.wkernel, wkl.hpad, wkl.wpad, wkl.hstride, wkl.wstride))
                        count += 1

    return final_workloads

In [ ]:
all_wkls = []
for model_id in model_ids:
    with torch.no_grad():
        extracted_wkls = extract_wkls(net,arch_mapping, model_id)
    all_wkls.extend(extracted_wkls)


In [ ]:
extracted_wkls